# 03 Product categories

**In:** `products_cl.csv` (folder `data/clean-data`, made by `01_data-cleaning.ipynb`).
**Out:** `products_categorised.csv` (folder `data/clean-data`), with 3 new columns:


- `sub_category`: 86 specific category names, e.g. *iPhone Cases*, *Apple Watch 42mm*, *iMac 27"*
- `grouped_category`: 12 big groups
- 1 extra column, `condition` (New / Open box / Refurbished / Second hand), taken from the start of the product name.


**Method:**
1. Use the `type` column for the 80 biggest types. Four of them are mixed and skipped, so this names 8,840 products (88.5%).

2. Find the expensive types outside the top 80 (mostly devices like iPhones and iPads).

3. Only after above, then use word searches on names, for what is still blank. For devices I also use a price break point.
4. One more name search for monitors. Everything still blank (mostly cheap open-box accessories) goes to *Open box & other (unsorted)*. Sorting those by hand-picked words would take dozens of searches for about 9% of products, which is beyond what the reporting needs.
5. Sanity check, then group the sub-categories.

**Decisions made before starting:**
- Type `1298` is **not** a product type. It is open-box returns of everything (monitors, cases, hard drives, iPads). I leave it blank in step 2. Its devices and monitors are found by name in step 4; its accessories stay unsorted. The new `condition` column keeps track of open-box items. Types `12355400`, `14365395` and `5403` are also mixed, so they are treated the same way.

- When two type codes hold the same thing (e.g. two MacBook Pro codes), they get the **same** sub-category name.

- Gadget type `11905404` is split: smartwatches and fitness trackers move to *Audio, Wearables & Media Players*.

- Wacom graphics tablets go to *Peripherals, Smart Home & Gadgets*: they are drawing pads for a computer, not standalone devices like an iPad. Watch straps and armbands go to *Cases, Cables & Accessories*.

## 1. Setup

In [1]:
import pandas as pd

In [2]:
# show prices with 2 decimals, and allow long tables
pd.set_option("display.float_format", lambda x: "%.2f" % x)
pd.set_option("display.max_rows", 1000)

In [ ]:
# products_cl.csv (my Drive: data/clean-data)
url = "https://drive.google.com/file/d/1FqemLQP6QNQidE9FQ80n9NvwwTIySosl/view?usp=drive_link"
path = "https://drive.google.com/uc?export=download&id=" + url.split("/")[-2]
products_cl = pd.read_csv(path)

In [4]:
products_cl.shape
# expected: (9992, 6)

(9992, 6)

In [5]:
# work on a copy, so products_cl stays untouched
products_df = products_cl.copy()

In [6]:
# new empty column, filled step by step below
products_df["sub_category"] = ""
products_df.head()

,sku,name,desc,price,in_stock,type,sub_category
0,RAI0007,Silver Rain Design mStand Support,Aluminum support compatible with all MacBook,59.99,1,8696,
1,APP0023,Apple Mac Keyboard Keypad Spanish,USB ultrathin keyboard Apple Mac Spanish.,59.00,0,13855401,
2,APP0025,Mighty Mouse Apple Mouse for Mac,mouse Apple USB cable.,59.00,0,1387,
3,APP0072,Apple Dock to USB Cable iPhone and iPod white,IPhone dock and USB Cable Apple iPod.,25.00,0,1230,
4,KIN0007,Mac Memory Kingston 2GB 667MHz DDR2 SO-DIMM,2GB RAM Mac mini and iMac (2006/07) MacBook Pr...,34.99,1,1364,


## 2. The 80 biggest types

In [7]:
# how many products the 80 biggest types cover, as a %
(products_df["type"].value_counts().head(80).sum() / products_df.shape[0] * 100).round(1)

np.float64(97.0)

**What we see:** the 80 biggest types hold 97.0% of all 9,992 products (the 60 biggest would hold 92.5%). But four of these types are mixed and get skipped, so the share this step really names is lower. The count after the assignments shows the real number.

In [8]:
# look inside each of the 80 biggest types:
# for each type, print how many products, the median price, and 5 example names
top_80_types = products_df["type"].value_counts().head(80).index

for t in top_80_types:
    products_of_type = products_df.loc[products_df["type"] == t]
    print(t, "|", len(products_of_type), "products | median price", products_of_type["price"].median())
    print(products_of_type["name"].sample(5, random_state=1).tolist())
    print()

11865403 | 1057 products | median price 39.0
['Leather Case Apple iPhone 6 / 6S Gray Storm', 'Stikgo Case + Support iPhone 8/7 Plus Red Car', 'Macally Snap-on Case iPhone 6 / 6S Metallic Black', 'Air Belkin iPhone Case Protect SheerForce 8 Plus / 7 Plus Rose Gold', 'Like new - Apple iPhone Leather Case Cover 7 Red Plus']

12175397 | 939 products | median price 1006.95
['Synology DS1817 + | 2GB RAM | 16TB (8x2TB) Seagate Iron Wolf', 'QNAP TVS-1271U-RP NAS with redundant source', 'DS218play Synology NAS Server | 16TB (2x8TB) Seagate Iron Wolf', 'Pack QNAP TS-451 + | 2GB RAM + WD Red 24TB', 'Synology DS716 + II | 8GB RAM Mac and PC Server Nas']

1298 | 783 products | median price 79.99
['Open - Wacom Bamboo Folio A4 Black', 'Open - LMP Bluetooth Keypad 2 Numeric Keypad for Magic Keyboard', '(Open) Messenger Zagg Folio Keyboard Folio iPad Air / Air 2 Black', 'Open - Tucano DUEINUNO iPhone Case 2 in 1 Gold 7', 'Open - Kit Transcend JetDrive expansion SSD 725 240GB Retina Macbook Pro 15 "fro

**What we see** (the names below come from reading this output):
- Most types are clear: `11865403` is all iPhone cases, `12175397` is all NAS servers, `1296` is monitors.
- Type `1298` is a mix of everything, all with "Open -" or "(Open)" in front. It's a **condition**, not a product type, so I skip it here.
- Some types that look like duplicates really differ: `24885185` is Apple Watch **38mm**, `24895185` is Apple Watch **42mm**. The same for iPhone 7 vs iPhone 7 Plus, and iMac 21.5" vs iMac 27".
- `1,02E+12` and `2158` are both MacBook Pro, with the same mix of 13"/15" and prices. No real difference, so both get the name *MacBook Pro*.
- Codes like `5,74E+15` are broken numbers (a spreadsheet turned long codes into scientific notation). The products inside are still consistent (`5,74E+15` is all iMac 27"), so I can still use them. This is a point for the data-collection slide.
- `1282` is a mix of Macs (MacBook Air, Mac mini, iMac, even Cinema Displays). I call it *Macs (mixed models)* for now and split it by name in step 4.
- Types 61 to 80 add 7 clean accessory types: `10230` laptop bags, `2434` Apple Watch cases, `1387` mice and trackpads, `10142` MacBook batteries, `14035403` iPad screen protectors, `12575403` iPad keyboard cases, `1276` PCIe cards. They also add 9 device types (iPhone 6s, 6s Plus, 8 Plus, iPad mini 4, iPad Pro 9.7", 10.5" and 12.9", plus `1716` and `1714`) and `1416`, which is software (Parallels, Adobe, Office).
- `12355400`, `14365395` and `5403` look mixed in the 5 examples. I check them in the next cell.

In [9]:
# 5 examples aren't enough for the types that look mixed: show all their names
products_df.loc[products_df["type"].isin(["1716", "1714", "12355400", "14365395", "5403"]), ["type", "name", "price"]].sort_values("type")

,type,name,price
23,12355400,Mobee Magic Charger Charger Magic Mouse,49.99
7771,12355400,Kensington Security Locking Station 2.0 Adapte...,120.01
4448,12355400,Macally Bamboo Cooling Stand Macbook,34.95
4228,12355400,X-Rite i1 Display monitor calibrator PRO,266.20
4226,12355400,X-Rite ColorMunki Display monitor calibrator,187.99
4164,12355400,Palette Color Cube Portable Scanner,199.99
4163,12355400,X-Rite ColorMunki Photo + GrafiLite Free,613.00
4162,12355400,X-Rite ColorMunki Photo Calibrator,526.35
3952,12355400,Macally UC3HUB USB Hub-C 3.1 to 4 USB-A,34.95
2267,12355400,Slot Adapter Kit security Kensington,16.49


**What we see:**
- `1716` is open-box iPhones of different models, so it becomes *iPhones (mixed models)*. `1714` is iPad Pro 12.9" plus some normal iPads, so it becomes *iPad Pro 12.9" & iPad*.
- `12355400` is a real mix: GoPro mounts, Mobee chargers, X-Rite colour calibrators, security locks, a laptop cooling stand.
- `14365395` mixes Lightning adapters and power adapters with car phone holders.
- `5403` is all cases, but for different things: MacBooks, iPhones, iPads, Apple Pencils, even a GoPro float.
- These three get no name here, like `1298`. Their products are sorted by name in step 4.

In [10]:
# sub-category for each of the top 80 types
# (76 here: types 1298, 12355400, 14365395 and 5403 are mixed, so they are skipped on purpose)

# storage and memory
products_df.loc[products_df["type"] == "11935397", "sub_category"] = "External Hard Drives"
products_df.loc[products_df["type"] == "12655397", "sub_category"] = "Internal Hard Drives"
products_df.loc[products_df["type"] == "12215397", "sub_category"] = "Internal SSDs"
products_df.loc[products_df["type"] == "1433", "sub_category"] = "SSD Upgrade Kits"
products_df.loc[products_df["type"] == "1364", "sub_category"] = "RAM Memory"
products_df.loc[products_df["type"] == "57445397", "sub_category"] = "USB Flash Drives & Memory Cards"
products_df.loc[products_df["type"] == "42945397", "sub_category"] = "Lightning Flash Drives"
products_df.loc[products_df["type"] == "12755395", "sub_category"] = "Drive Mounting & Install Kits"

# iPhones
products_df.loc[products_df["type"] == "51601716", "sub_category"] = "iPhone SE & 5S"
products_df.loc[products_df["type"] == "85641716", "sub_category"] = "iPhone 7"
products_df.loc[products_df["type"] == "85651716", "sub_category"] = "iPhone 7 Plus"
products_df.loc[products_df["type"] == "24821716", "sub_category"] = "iPhone 6s"                 # types 61-80
products_df.loc[products_df["type"] == "24811716", "sub_category"] = "iPhone 6s Plus"            # types 61-80
products_df.loc[products_df["type"] == "113281716", "sub_category"] = "iPhone 8 Plus"            # types 61-80
products_df.loc[products_df["type"] == "1716", "sub_category"] = "iPhones (mixed models)"        # types 61-80

# cases and protection
products_df.loc[products_df["type"] == "11865403", "sub_category"] = "iPhone Cases"
products_df.loc[products_df["type"] == "12635403", "sub_category"] = "iPad Cases & Covers"
products_df.loc[products_df["type"] == "13835403", "sub_category"] = "Laptop Sleeves & Cases"
products_df.loc[products_df["type"] == "1392", "sub_category"] = "Laptop Backpacks & Bags"
products_df.loc[products_df["type"] == "5,49E+11", "sub_category"] = "Battery Cases"
products_df.loc[products_df["type"] == "13555403", "sub_category"] = "Screen Protectors"
products_df.loc[products_df["type"] == "2449", "sub_category"] = "Apple Watch Straps"
products_df.loc[products_df["type"] == "5405", "sub_category"] = "Sport Armbands"
products_df.loc[products_df["type"] == "2434", "sub_category"] = "Apple Watch Cases"             # types 61-80
products_df.loc[products_df["type"] == "14035403", "sub_category"] = "Screen Protectors"         # types 61-80
products_df.loc[products_df["type"] == "12575403", "sub_category"] = "iPad Keyboard Cases"       # types 61-80
products_df.loc[products_df["type"] == "10230", "sub_category"] = "Laptop Backpacks & Bags"      # types 61-80

# cables, charging and stands
products_df.loc[products_df["type"] == "1230", "sub_category"] = "Lightning Cables"
products_df.loc[products_df["type"] == "1325", "sub_category"] = "Video Cables, Adapters & Locks"
products_df.loc[products_df["type"] == "12585395", "sub_category"] = "USB-C Adapters & Hubs"
products_df.loc[products_df["type"] == "1515", "sub_category"] = "Power Banks"
products_df.loc[products_df["type"] == "13615399", "sub_category"] = "Phone Chargers & Docks"
products_df.loc[products_df["type"] == "13005399", "sub_category"] = "MacBook Chargers & Batteries"
products_df.loc[products_df["type"] == "10142", "sub_category"] = "MacBook Chargers & Batteries" # types 61-80
products_df.loc[products_df["type"] == "24215399", "sub_category"] = "Apple Watch Chargers & Docks"
products_df.loc[products_df["type"] == "5395", "sub_category"] = "Power Strips & Adapters"
products_df.loc[products_df["type"] == "5720", "sub_category"] = "Phone Stands & Car Mounts"
products_df.loc[products_df["type"] == "1216", "sub_category"] = "iPad Stands, Mounts & Kiosks"
products_df.loc[products_df["type"] == "8696", "sub_category"] = "Laptop & Desktop Stands"

# peripherals and gadgets
products_df.loc[products_df["type"] == "13855401", "sub_category"] = "Keyboards"
products_df.loc[products_df["type"] == "1387", "sub_category"] = "Mice & Trackpads"            # types 61-80
products_df.loc[products_df["type"] == "1229", "sub_category"] = "Styluses, Smart Pens & Presenters"
products_df.loc[products_df["type"] == "12995397", "sub_category"] = "Hubs, Docks & Drive Enclosures"
products_df.loc[products_df["type"] == "1334", "sub_category"] = "Routers & Network Switches"
products_df.loc[products_df["type"] == "9094", "sub_category"] = "Security Cameras"
products_df.loc[products_df["type"] == "1276", "sub_category"] = "PCIe & Expansion Cards"       # types 61-80
products_df.loc[products_df["type"] == "11905404", "sub_category"] = "Smart Home & Gadgets"   # split later (step 4)

# monitors
products_df.loc[products_df["type"] == "1296", "sub_category"] = "Monitors"

# audio and wearables
products_df.loc[products_df["type"] == "5384", "sub_category"] = "Headphones"
products_df.loc[products_df["type"] == "5398", "sub_category"] = "Speakers"
products_df.loc[products_df["type"] == "11821715", "sub_category"] = "iPod"
products_df.loc[products_df["type"] == "24885185", "sub_category"] = "Apple Watch 38mm"
products_df.loc[products_df["type"] == "24895185", "sub_category"] = "Apple Watch 42mm"

# NAS and servers
products_df.loc[products_df["type"] == "12175397", "sub_category"] = "NAS Servers"
products_df.loc[products_df["type"] == "1404", "sub_category"] = "NAS Expansion Units & Cards"

# iPads and tablets
products_df.loc[products_df["type"] == "12141714", "sub_category"] = 'iPad Air 2 & iPad Pro 9.7"'
products_df.loc[products_df["type"] == "13621714", "sub_category"] = "iPad mini 4"               # types 61-80
products_df.loc[products_df["type"] == "51871714", "sub_category"] = 'iPad Pro 9.7"'             # types 61-80
products_df.loc[products_df["type"] == "106431714", "sub_category"] = 'iPad Pro 10.5"'           # types 61-80
products_df.loc[products_df["type"] == "51861714", "sub_category"] = 'iPad Pro 12.9"'            # types 61-80
products_df.loc[products_df["type"] == "1714", "sub_category"] = 'iPad Pro 12.9" & iPad'         # types 61-80
products_df.loc[products_df["type"] == "1405", "sub_category"] = "Wacom Graphics Tablets"

# Macs (two MacBook Pro codes get the same name: same products)
products_df.loc[products_df["type"] == "1,02E+12", "sub_category"] = "MacBook Pro"
products_df.loc[products_df["type"] == "2158", "sub_category"] = "MacBook Pro"
products_df.loc[products_df["type"] == "2,17E+11", "sub_category"] = "MacBook Pro & Air (Like new)"
products_df.loc[products_df["type"] == "5,39E+11", "sub_category"] = 'MacBook 12"'
products_df.loc[products_df["type"] == "2,16E+11", "sub_category"] = 'iMac 21.5"'
products_df.loc[products_df["type"] == "5,74E+15", "sub_category"] = 'iMac 27"'
products_df.loc[products_df["type"] == "118692158", "sub_category"] = "iMac Pro"
products_df.loc[products_df["type"] == "21632158", "sub_category"] = "Mac Pro"
products_df.loc[products_df["type"] == "1282", "sub_category"] = "Macs (mixed models)"   # split later (step 4)

# repairs
products_df.loc[products_df["type"] == "1,44E+11", "sub_category"] = "Repair Services"
products_df.loc[products_df["type"] == "21485407", "sub_category"] = "iPhone Spare Parts"
products_df.loc[products_df["type"] == "12645406", "sub_category"] = "iPhone Repair Tools & Kits"
products_df.loc[products_df["type"] == "14305406", "sub_category"] = "Mac Repair Tools & Kits"

# software
products_df.loc[products_df["type"] == "1416", "sub_category"] = "Software"                      # types 61-80

# types 1298, 12355400, 14365395 and 5403 (mixed): left blank on purpose, sorted by name in step 4

In [11]:
# how many products still have no sub-category
(products_df["sub_category"] == "").sum()

np.int64(1152)

**What we see:** 1,152 products are still blank. So the type codes name 8,840 products (9,992 − 1,152), which is 88.5% of the catalogue. Of the 1,152 blank ones, 783 are the open-box type `1298`, 67 come from the three mixed types, 46 have no type at all, the rest are small types.

## 3. Expensive types outside the top 80

In [12]:
# types we haven't used yet (still blank and not in the top 80)
not_done = products_df.loc[(products_df["sub_category"] == "") & ~(products_df["type"].isin(top_80_types))]

# one row per type: number of products, median price, one example name. Most expensive first
not_done.groupby("type").agg(products=("sku", "count"),
                             median_price=("price", "median"),
                             example=("name", "first")).sort_values("median_price", ascending=False).head(25)

,products,median_price,example
type,,,
51912158,1,2805.59,"Open - Apple MacBook Pro 15 ""Core i7 Touch Bar..."
"9,29E+11",1,2799.00,"Apple MacBook Pro 13 ""with Touch Bar | Keyboar..."
51882158,10,2229.00,"Apple iMac 27 ""Core i7 Retina 5K 4GHz | 32GB |..."
"5,43E+15",14,1199.00,"Second hand - Apple iMac 20 ""Core 2 Duo 24GHz ..."
"5,72E+15",6,1199.00,"Second hand - Apple iMac 215 ""Core i5 Quad-Cor..."
113271716,4,1159.00,Apple iPhone 64GB Space Gray X
51902158,1,1099.00,"Apple Macbook Air 13 ""i5 16GHz | 8GB RAM | 128..."
21571716,5,859.00,Like new - Apple iPhone 6 Plus 16GB Space Gray
5185,1,819.00,Apple Watch the 1st Gen. 42mm Stainless Steel ...


**What we see:** the types with a median price above about €250 are almost all devices: iPhone X, iPhone 8, iPhone 6, Mac mini, second-hand iMacs, iPad mini 4, iPad Air 2. Below that it turns into iPods, Apple TVs and accessories, which the name searches in step 4 catch. So I stop at €250 (20 types). One type needs a closer look first.

In [13]:
# 12051714 looks like iPad Air, but its median is lower than the others: check its names
products_df.loc[products_df["type"] == "12051714", ["type", "name", "price"]]

,type,name,price
6608,12051714,Speaker repair inside iPad Air,69.99
7092,12051714,Open - Apple iPad Air Wi-Fi 16GB Silver,389.00
7093,12051714,Second hand - Apple iPad 16GB Wi-Fi Air Space ...,389.00


**What we see:** `12051714` is iPad Air, with one repair service in it (fixed in the sanity check, step 5).

In [14]:
# sub-category for the expensive types (median price above €250)

# iPhones
products_df.loc[products_df["type"] == "21561716", "sub_category"] = "iPhone 6"
products_df.loc[products_df["type"] == "21571716", "sub_category"] = "iPhone 6 Plus"
products_df.loc[products_df["type"] == "113291716", "sub_category"] = "iPhone 8"
products_df.loc[products_df["type"] == "113271716", "sub_category"] = "iPhone X"

# iPads
products_df.loc[products_df["type"] == "113851714", "sub_category"] = "iPad"
products_df.loc[products_df["type"] == "12051714", "sub_category"] = "iPad Air"
products_df.loc[products_df["type"] == "42931714", "sub_category"] = "iPad Air 2"
products_df.loc[products_df["type"] == "12031714", "sub_category"] = "iPad mini 2"
products_df.loc[products_df["type"] == "24861714", "sub_category"] = "iPad mini 4"

# Macs
products_df.loc[products_df["type"] == "51912158", "sub_category"] = "MacBook Pro"
products_df.loc[products_df["type"] == "9,29E+11", "sub_category"] = "MacBook Pro"
products_df.loc[products_df["type"] == "51902158", "sub_category"] = "MacBook Air"
products_df.loc[products_df["type"] == "5,43E+15", "sub_category"] = 'iMac 20"'
products_df.loc[products_df["type"] == "51882158", "sub_category"] = "iMac (mixed sizes)"
products_df.loc[products_df["type"] == "5,72E+15", "sub_category"] = "iMac (mixed sizes)"
products_df.loc[products_df["type"] == "5,44E+11", "sub_category"] = "Mac mini"
products_df.loc[products_df["type"] == "5,45E+15", "sub_category"] = "Mac mini"
products_df.loc[products_df["type"] == "21622158", "sub_category"] = "Mac mini"

# others
products_df.loc[products_df["type"] == "5185", "sub_category"] = "Apple Watch 42mm"
products_df.loc[products_df["type"] == "1280", "sub_category"] = "NAS Expansion Units & Cards"

In [15]:
(products_df["sub_category"] == "").sum()

np.int64(1054)

**What we see:** 1,054 products are still blank (783 open box, 67 from the three mixed types, 46 with no type, 158 from small types). From here on I only use name searches.

## 4. Name searches (only for products that are still blank)

Every search below has `products_df["sub_category"] == ""` in it, so it never overwrites a sub-category from steps 2 and 3. The **order matters**: a product is taken by the first search it matches.

In [16]:
# shortcut so the lines below are shorter
name = products_df["name"]

### 4a. Devices: word search + price break point

In [17]:
# still blank + "iphone" in the name, cheapest first
products_df.loc[(products_df["sub_category"] == "") & (name.str.contains("iphone", case=False)), ["name", "price"]].sort_values("price")

,name,price
4502,iFixit part adhesive strips Battery iPhone 6 / 6S,5.95
4499,iFixit part adhesive strips Battery iPhone 6 Plus,5.99
4506,iFixit part adhesive strips Battery iPhone 5s ...,5.99
6482,Open - Tucano Sottile iPhone Case SE / 5s / 5 ...,7.90
5707,(Open) SwitchEasy iPhone Case 7 0.35,9.99
6319,Open - SwitchEasy iPhone Case 0.35 7 Blanco,9.99
6188,(Open) SwitchEasy iPhone Case 0.35 7 Plus,9.99
3099,Moxie Car Support for compatible iPhone Volante,9.99
8112,Open - Wowewa Protector Tempered Glass iPhone ...,9.99
3959,(Open) NewerTech iPhone Kit tools,12.99


**What we see:** up to €150 it's all cases, chargers and spare parts. Then there's a clear jump to €409, where the real iPhones start. So the break point is €300.

In [18]:
# still blank + "iphone" in the name + price over €300 → iPhones
products_df.loc[(products_df["sub_category"] == "")
              & (name.str.contains("iphone", case=False))
              & (products_df["price"] > 300), "sub_category"] = "iPhones (mixed models)"

In [19]:
# same idea for Macs. First try the simple search: still blank + Mac words, most expensive first
products_df.loc[(products_df["sub_category"] == "")
              & (name.str.contains("imac|mac mini|mac pro|macbook", case=False)), ["name", "price"]].sort_values("price", ascending=False).head(15)

,name,price
6405,"Open - Apple MacBook Pro 15 ""Core i7 Touch Bar...",3799.00
6467,"Open - Apple MacBook Pro 15 ""Core i7 Touch Bar...",3179.00
6374,"Open - Apple iMac 27 ""Core i7 Retina 5K 4GHz |...",3169.00
7911,"Like new - Apple MacBook Pro 15 ""Core i7 Touch...",2939.00
7822,"Open - Apple MacBook Pro 13 ""with Touch Bar 33...",2799.00
8092,"Open - Apple iMac 27 ""Core i5 3.3GHz Retina 5K...",2749.00
7905,"Like new - Apple MacBook Pro 15 ""Core i7 Touch...",2699.00
7831,"Like new - Apple MacBook Pro 15 ""Core i7 Touch...",2699.00
6501,"Open - Apple MacBook Pro 13 ""with Touch Bar 33...",2599.00
7830,"Open - Apple iMac 27 ""Core i5 Retina 5K 35GHz ...",2305.59


**What we see:** the simple search doesn't work for Macs. SSD upgrade kits *for* a MacBook cost up to €437, more than an old Mac mini. Price alone can't separate them.

The fix: real Macs have "Apple" + the model at the **start** of the name (after "Open - " or "Like new - " at most). `^.{0,15}` means "the start of the name, then at most 15 other characters". `(?:...)` just groups the choices, like brackets in maths.

In [20]:
# Macs: "Apple" + model near the start of the name, cheapest first
products_df.loc[(products_df["sub_category"] == "")
              & (name.str.contains(r"^.{0,15}apple (?:imac|mac mini|mac pro|macbook)", case=False)), ["name", "price"]].sort_values("price")

,name,price
8424,Open - Apple MacBook Pro MagSafe 85W Charger,89.00
7032,Second hand - Apple Mac mini Core 2 Duo 226Ghz...,599.00
3304,Open - Apple Mac mini Core i5 14GHz | 8GB RAM ...,675.59
6649,Open - Apple Mac mini Core i5 26GHz | 8GB RAM ...,805.59
7690,"Open - Apple MacBook Air 13 ""i5 16 Ghz | 8GB R...",1099.00
6931,"Open - Apple MacBook Pro 133 ""Core i5 25GHz |...",1199.00
7903,"Like new - Apple MacBook Pro 13 ""Core i5 25GHz...",1199.00
5051,"Open - Apple MacBook Air 13 ""i7 22 Ghz | 8GB R...",1219.00
5715,(Open) Apple Mac mini Core i5 28GHz | 16GB RAM...,1339.00
6288,"Open - Apple MacBook Air 13 ""i5 16 Ghz | 8GB R...",1349.00


**What we see:** now only one cheap item is left (an €89 charger), then a jump to €599 for the cheapest Mac mini. Break point €300.

In [21]:
products_df.loc[(products_df["sub_category"] == "")
              & (name.str.contains(r"^.{0,15}apple (?:imac|mac mini|mac pro|macbook)", case=False))
              & (products_df["price"] > 300), "sub_category"] = "Macs (mixed models)"

In [22]:
# iPads and Apple Watches: same start-of-name search, cheapest first
products_df.loc[(products_df["sub_category"] == "")
              & (name.str.contains(r"^.{0,15}apple ipad|^.{0,15}apple watch", case=False)), ["name", "price"]].sort_values("price")

,name,price
6945,Open - Satechi Apple Watch Support Aluminum Si...,34.99
21,Apple iPad Camera Connection connector,35.00
7958,Like new - Apple iPad Camera Connection Kit,35.00
7803,Like new - Apple iPad mini Smart Cover Black,45.00
4783,(Open) Nomad Apple Watch Pod Pro iPhone Charge...,79.95
7804,"Like new - Apple iPad Silicone Case Pro 129 ""C...",89.00
4994,Smart Apple iPad Keyboard Case with Keyboard P...,169.00
6794,Open - Air Apple iPad 2 Wi-Fi 32GB Dorado,429.00
6076,(Open) Apple iPad Air 2 Wi-Fi 32GB Silver,429.00
6249,Open - Apple Watch 42mm Aluminum Case Series 2...,469.00


**What we see:** iPad accessories go up to €169 (keyboard case), then real iPads start at €429. The one real Apple Watch costs €469; the accessories stay under €80. Break point €300 for iPads and €200 for the watch.

In [23]:
products_df.loc[(products_df["sub_category"] == "")
              & (name.str.contains(r"^.{0,15}apple ipad", case=False))
              & (products_df["price"] > 300), "sub_category"] = "iPads (mixed models)"

products_df.loc[(products_df["sub_category"] == "")
              & (name.str.contains(r"^.{0,15}apple watch", case=False))
              & (products_df["price"] > 200), "sub_category"] = "Apple Watch 42mm"   # the only one is 42mm

In [24]:
(products_df["sub_category"] == "").sum()

np.int64(969)

**What we see:** 969 still blank. The devices are done (41 iPhones, 32 Macs, 11 iPads, 1 watch). What's left is accessories, mostly open-box ones.

### 4b. Monitors, then the rest stays unsorted

The devices are done. Monitors are the other expensive product that is still blank (open-box monitors in type `1298`), so they get one more search. Everything else stays unsorted on purpose.

In [25]:
# still blank + "monitor" in the name, most expensive first
products_df.loc[(products_df["sub_category"] == "") & (name.str.contains("monitor", case=False)), ["name", "price"]].sort_values("price", ascending=False)

,name,price
6204,"Open - LG 38UC99-W Monitor 38 ""UHD 4K USB 3.0 ...",1299.00
5418,"(Open) Dell Monitor P4317Q 425 """" IPS 4K DP mD...",1166.44
1828,"(Open) Apple Thunderbolt Display 27 ""Monitor Mac",1149.00
4836,"Open - LG 34UC98-W Monitor 34 ""Curved QHD Thun...",1099.00
5577,"Open - 31MU97Z-B LG Monitor 31 ""4K pivotable T...",1099.00
4949,"Open - Dell UP2715K IPS Monitor 27 ""5K",1040.60
1529,"Open - 34UM95 LG LCD Monitor 34 ""Ultra WQHD re...",919.00
8670,"Open - LG 43UD79-B Monitor 425 ""4K 72% NTSC US...",845.99
4556,"Open - Dell P2715Q 27 ""IPS 4K UHD Monitor sRGB...",745.36
6238,"Open - LG 34UM88-P Monitor 34 ""IPS QHD Thunder...",699.00


**What we see:** mostly open-box LG, Dell and BenQ monitors (€115 to €1,299). But "monitor" also catches things that are not monitors:
- **activity monitors** (Jawbone fitness trackers)
- **baby monitor cameras** (D-Link)
- **monitor calibrators** (X-Rite)

So these three words are excluded with `~`.

In [26]:
# still blank + "monitor", but NOT activity monitors, baby monitor cameras or calibrators → Monitors
# ~ means "NOT"
products_df.loc[(products_df["sub_category"] == "")
              & (name.str.contains("monitor", case=False))
              & ~(name.str.contains("activity|camera|calibrator", case=False)), "sub_category"] = "Monitors"

In [27]:
(products_df["sub_category"] == "").sum()

np.int64(931)

**What we see:** 931 products are still blank.

In [28]:
# what is still blank: how many are open box (type 1298), and their prices
(products_df.loc[products_df["sub_category"] == "", "type"] == "1298").value_counts()

type
True     698
False    233
Name: count, dtype: int64

In [29]:
# price spread of what is still blank
products_df.loc[products_df["sub_category"] == "", "price"].describe()

count    931.00
mean     126.08
std      224.21
min        4.90
25%       34.99
50%       60.99
75%      129.99
max     3199.90
Name: price, dtype: float64

In [30]:
# a random look at what is still blank
products_df.loc[products_df["sub_category"] == "", ["name", "price", "type"]].sample(30, random_state=1)

,name,price,type
4295,"iFixit Cable HDD with Bracket MacBook Pro 13 ""...",45.99,54085407
4499,iFixit part adhesive strips Battery iPhone 6 Plus,5.99,5407
5960,Open - Air Protect SheerForce Belkin iPhone Ca...,19.99,1298
4791,(Open) LaCie CloudBox 3TB Hard Drive,149.99,1298
4694,Front Panel Kit Full iFixit iPad 4 White,49.95,21535407
584,AppleCare Protection Plan Mac mini,89.00,1231
9793,Open - Synology DS218 + NAS server Mac and PC,344.99,NaN
3576,(Open) Western Digital Elements Desktop USB 3....,124.99,1298
8091,Open - Sandisk iXpand Lightning to USB 3.0 128GB,99.99,1298
7044,Open - 25 or OWC Dual Dock Station USB 3.1 HDD 35,96.99,1298


**What we see:**
- 698 of the 931 are open-box products (type `1298`). The other 233 come from the three mixed types and small types.
- They are mostly cheap: median €61, half of them between €35 and €130.
- The random look shows iPhone cases, cables and adapters, hard drives, SSDs, iFixit parts, a few NAS servers and Wacom tablets.

I put them all in one group, *Open box & other (unsorted)*. That's about 9% of the catalogue. When we look at sales, I'll check how much **revenue** this group has, so we know if leaving it unsorted matters. It's also a point for the data-collection slide: type `1298` mixes the condition (open box) with the product type, so the data can't tell what these products are without reading their names.

In [31]:
products_df.loc[products_df["sub_category"] == "", "sub_category"] = "Open box & other (unsorted)"

### 4c. Split the two mixed sub-categories

In [32]:
# "Macs (mixed models)": what's in it
products_df.loc[products_df["sub_category"] == "Macs (mixed models)", "name"].str[:40].value_counts().head(20)

name
Apple iMac 27 "Core i5 3.2GHz Retina 5K     29
Second hand - Apple iMac 215 "Core i5 Qu    14
Apple iMac 21.5 "Core i5 2.8GHz | 8GB |     12
Apple iMac 21.5 "Core i5 2.8GHz | 16GB |    12
Apple MacBook Pro Retina 13 "i5 29 Ghz |     8
Apple iMac 21.5 "Core i7 33GHz 4K Retina     8
Apple MacBook Retina 12 "Core M7 13GHz |     8
Apple Mac mini Core i7 3GHz | 16GB RAM |     7
Apple iMac 215 "Core i5 16GHz | 8GB | 1T     6
Apple iMac 215 "Core i5 16GHz | 16GB | 1     6
Apple MacBook Pro Retina 13 "Core i7 3GH     6
Apple MacBook Pro Retina 13 "Core i5 29G     6
Apple iMac 21.5 "Core i5 31 GHz Retina d     6
Apple Mac Pro Quad-Core 37GHz | 32GB RAM     6
Apple Mac Pro six cores 35GHz | 32GB RAM     6
Apple Mac Pro six cores 35GHz | 64GB RAM     6
Apple Mac Pro Quad-Core 37GHz | 12GB RAM     6
Apple Mac Pro six cores 35GHz | 16GB RAM     6
Apple MacBook Air 13 "i7 22 Ghz | 8GB RA     6
Apple MacBook Air 13 "i5 16 Ghz | 8GB RA     6
Name: count, dtype: int64

**What we see:** MacBook Pros, MacBook Airs, Mac minis, iMacs, Mac Pros, and also Apple Cinema Displays (those are monitors, not computers). The searches below give each one its real name. "macbook pro" and "macbook air" go **before** plain "macbook", otherwise every MacBook Pro would become MacBook 12".

In [33]:
products_df.loc[(products_df["sub_category"] == "Macs (mixed models)") & (name.str.contains("cinema display", case=False)), "sub_category"] = "Monitors"
products_df.loc[(products_df["sub_category"] == "Macs (mixed models)") & (name.str.contains("macbook pro", case=False)), "sub_category"] = "MacBook Pro"
products_df.loc[(products_df["sub_category"] == "Macs (mixed models)") & (name.str.contains("macbook air", case=False)), "sub_category"] = "MacBook Air"
products_df.loc[(products_df["sub_category"] == "Macs (mixed models)") & (name.str.contains("macbook", case=False)), "sub_category"] = 'MacBook 12"'
products_df.loc[(products_df["sub_category"] == "Macs (mixed models)") & (name.str.contains("mac mini", case=False)), "sub_category"] = "Mac mini"
products_df.loc[(products_df["sub_category"] == "Macs (mixed models)") & (name.str.contains("imac", case=False)), "sub_category"] = "iMac (mixed sizes)"
products_df.loc[(products_df["sub_category"] == "Macs (mixed models)") & (name.str.contains("mac pro", case=False)), "sub_category"] = "Mac Pro"

In [34]:
# check: nothing should be left in "Macs (mixed models)"
(products_df["sub_category"] == "Macs (mixed models)").sum()

np.int64(0)

In [35]:
# "Smart Home & Gadgets": look for smartwatches and fitness trackers
products_df.loc[(products_df["sub_category"] == "Smart Home & Gadgets")
              & (name.str.contains(r"watch|fitbit|activit|tracker|jawbone|misfit|garmin|pebble|mi band|bracelet|\bband\b", case=False)), ["name", "price"]].sort_values("price")

,name,price
1506,Runtastic Bracelet LED Safety Reflector,14.99
446,Replacement fitbit Flex 3 bracelets colors Size S,25.99
445,Replacement Fitbit Flex 3 colors bracelet size L,25.99
1477,Runtastic Bracelets Set color (yellow orange a...,29.99
1486,Runtastic Bracelets Set Color (Green Gray and ...,29.99
2035,Jawbone UP Move Pack 3 bracelets Thick White /...,29.99
4087,1s Xiaomi Mi Band Black Bracelet Monitor activity,39.99
4762,Monitor activity Withings Go Black,49.95
2033,Jawbone UP Move Activity Monitor Blue,49.99
2034,Jawbone UP Move Activity Monitor PÌ¼rpura,49.99


**What we see:** mostly Fitbit, Jawbone, Withings and Nokia trackers and watches. But the search also catches smart **scales** (Fitbit Aria, iHealth) and a running reflector. Those stay as gadgets, so I exclude them.

In [36]:
products_df.loc[(products_df["sub_category"] == "Smart Home & Gadgets")
              & (name.str.contains(r"watch|fitbit|activit|tracker|jawbone|misfit|garmin|pebble|mi band|bracelet|\bband\b", case=False))
              & ~(name.str.contains("scale|polaroid|reflector", case=False)), "sub_category"] = "Smartwatches & Fitness Trackers"
# ~ means "NOT": products whose name does NOT contain these words

In [37]:
(products_df["sub_category"] == "Smartwatches & Fitness Trackers").sum()

np.int64(69)

## 5. Sanity check: cheap products inside device sub-categories

In [38]:
# device sub-categories should not contain anything under €150
devices = ["iPhone SE & 5S", "iPhone 6", "iPhone 6 Plus", "iPhone 6s", "iPhone 6s Plus", "iPhone 7", "iPhone 7 Plus",
           "iPhone 8", "iPhone 8 Plus", "iPhone X", "iPhones (mixed models)",
           "iPad", "iPad Air", "iPad Air 2", 'iPad Air 2 & iPad Pro 9.7"', "iPad mini 2", "iPad mini 4",
           'iPad Pro 9.7"', 'iPad Pro 10.5"', 'iPad Pro 12.9"', 'iPad Pro 12.9" & iPad', "iPads (mixed models)",
           "MacBook Pro", "MacBook Air", 'MacBook 12"', "MacBook Pro & Air (Like new)", 'iMac 20"', 'iMac 21.5"',
           'iMac 27"', "iMac (mixed sizes)", "iMac Pro", "Mac mini", "Mac Pro", "Apple Watch 38mm", "Apple Watch 42mm"]

products_df.loc[products_df["sub_category"].isin(devices) & (products_df["price"] < 150), ["name", "price", "sub_category"]]

,name,price,sub_category
4519,iFixit piece Home button iPhone 5s Space Gray,29.95,iPhone SE & 5S
6608,Speaker repair inside iPad Air,69.99,iPad Air


**What we see:** two products don't belong: an iFixit home button (€29.95) that came with type `51601716`, and an iPad Air speaker repair (€69.99) from type `12051714`. The type codes put them in the wrong place, so I move them.

In [39]:
products_df.loc[(products_df["sub_category"] == "iPhone SE & 5S") & (products_df["price"] < 150), "sub_category"] = "iPhone Spare Parts"
products_df.loc[(products_df["sub_category"] == "iPad Air") & (products_df["price"] < 150), "sub_category"] = "Repair Services"

## 6. Condition column (from the start of the name)

In [40]:
# everything is "New" unless the name starts with a condition word
products_df["condition"] = "New"
products_df.loc[name.str.contains(r"^\(?open", case=False), "condition"] = "Open box"   # "Open - ..." or "(Open) ..."
products_df.loc[name.str.contains(r"^(?:like new|very good|excellent)", case=False), "condition"] = "Refurbished"
products_df.loc[name.str.contains(r"^second hand", case=False), "condition"] = "Second hand"
products_df["condition"].value_counts()

condition
New            8398
Open box       1185
Refurbished     318
Second hand      91
Name: count, dtype: int64

**What we see:** 8,398 new, 1,185 open box, 318 refurbished, 91 second hand. So about 1 in 6 products in the catalogue is not new. Open box covers more than type `1298`: other types also contain open-box items.

## 7. Grouped category

In [41]:
products_df["grouped_category"] = ""

# storage
products_df.loc[products_df["sub_category"].isin(["External Hard Drives", "Internal Hard Drives", "Internal SSDs", "SSD Upgrade Kits",
    "RAM Memory", "USB Flash Drives & Memory Cards", "Lightning Flash Drives", "Drive Mounting & Install Kits"]), "grouped_category"] = "Storage & Memory"

# phones
products_df.loc[products_df["sub_category"].isin(["iPhone SE & 5S", "iPhone 6", "iPhone 6 Plus", "iPhone 6s", "iPhone 6s Plus",
    "iPhone 7", "iPhone 7 Plus", "iPhone 8", "iPhone 8 Plus", "iPhone X", "iPhones (mixed models)"]), "grouped_category"] = "iPhones & Smartphones"

# cases, cables, charging, stands, straps
products_df.loc[products_df["sub_category"].isin(["iPhone Cases", "iPad Cases & Covers", "iPad Keyboard Cases", "Laptop Sleeves & Cases",
    "Laptop Backpacks & Bags", "Battery Cases", "Screen Protectors", "Apple Watch Cases", "Apple Watch Straps", "Sport Armbands",
    "Lightning Cables", "Video Cables, Adapters & Locks", "USB-C Adapters & Hubs", "Power Banks", "Phone Chargers & Docks",
    "MacBook Chargers & Batteries", "Apple Watch Chargers & Docks", "Power Strips & Adapters", "Phone Stands & Car Mounts",
    "iPad Stands, Mounts & Kiosks", "Laptop & Desktop Stands", "Cleaning Kits"]), "grouped_category"] = "Cases, Cables & Accessories"

# peripherals and gadgets (Wacom tablets are drawing pads for a computer, so they go here)
products_df.loc[products_df["sub_category"].isin(["Keyboards", "Mice & Trackpads", "Styluses, Smart Pens & Presenters", "Wacom Graphics Tablets",
    "Hubs, Docks & Drive Enclosures", "PCIe & Expansion Cards", "External Optical Drives", "Routers & Network Switches",
    "Security Cameras", "Smart Home & Gadgets", "Action Cameras & Accessories", "Microphones & Music Gear"]), "grouped_category"] = "Peripherals, Smart Home & Gadgets"

# monitors
products_df.loc[products_df["sub_category"].isin(["Monitors"]), "grouped_category"] = "Monitors & Displays"

# audio and wearables
products_df.loc[products_df["sub_category"].isin(["Headphones", "Speakers", "iPod", "Apple TV & Remotes",
    "Apple Watch 38mm", "Apple Watch 42mm", "Smartwatches & Fitness Trackers"]), "grouped_category"] = "Audio, Wearables & Media Players"

# NAS
products_df.loc[products_df["sub_category"].isin(["NAS Servers", "NAS Expansion Units & Cards"]), "grouped_category"] = "NAS & Servers"

# iPads and tablets
products_df.loc[products_df["sub_category"].isin(["iPad", "iPad Air", "iPad Air 2", 'iPad Air 2 & iPad Pro 9.7"', "iPad mini 2", "iPad mini 4",
    'iPad Pro 9.7"', 'iPad Pro 10.5"', 'iPad Pro 12.9"', 'iPad Pro 12.9" & iPad', "iPads (mixed models)"]), "grouped_category"] = "iPads & Tablets"

# Macs
products_df.loc[products_df["sub_category"].isin(["MacBook Pro", "MacBook Air", 'MacBook 12"', "MacBook Pro & Air (Like new)",
    'iMac 20"', 'iMac 21.5"', 'iMac 27"', "iMac (mixed sizes)", "iMac Pro", "Mac mini", "Mac Pro"]), "grouped_category"] = "Mac & Computers"

# repairs, spare parts and repair services
products_df.loc[products_df["sub_category"].isin(["Repair Services", "iPhone Spare Parts", "iPhone Repair Tools & Kits",
    "Mac Repair Tools & Kits", "AppleCare & Warranties"]), "grouped_category"] = "Repairs, Parts & Services"

# others
products_df.loc[products_df["sub_category"].isin(["Software"]), "grouped_category"] = "Others"

# not sorted on purpose (mostly open-box accessories, see step 4b)
products_df.loc[products_df["sub_category"].isin(["Open box & other (unsorted)"]), "grouped_category"] = "Unsorted (mostly open box)"

In [42]:
# check: no product should be without a group (expected: 0)
(products_df["grouped_category"] == "").sum()

np.int64(0)

## 8. Final checks

In [43]:
products_df["grouped_category"].value_counts()

grouped_category
Cases, Cables & Accessories          3023
Storage & Memory                     1426
Mac & Computers                      1159
NAS & Servers                         975
Unsorted (mostly open box)            931
Peripherals, Smart Home & Gadgets     899
Audio, Wearables & Media Players      600
Repairs, Parts & Services             279
iPhones & Smartphones                 261
Monitors & Displays                   229
iPads & Tablets                       186
Others                                 24
Name: count, dtype: int64

In [44]:
# price spread per group: do the products in each group belong together price-wise?
products_df.groupby("grouped_category")["price"].describe().sort_values("50%", ascending=False)

,count,mean,std,min,25%,50%,75%,max
grouped_category,,,,,,,,
Mac & Computers,1159.00,3294.74,2334.29,309.99,1829.00,2689.00,3689.00,15339.00
NAS & Servers,975.00,1174.33,843.25,9.99,601.16,984.95,1511.29,5894.49
iPhones & Smartphones,261.00,760.54,208.19,409.00,639.00,769.00,889.00,1329.00
iPads & Tablets,186.00,780.33,291.98,239.00,529.00,729.00,1009.00,1449.00
Monitors & Displays,229.00,520.65,373.97,104.99,231.11,399.99,717.53,2328.04
Others,24.00,353.39,357.03,49.99,85.24,290.25,435.45,1599.90
"Audio, Wearables & Media Players",600.00,256.01,224.66,3.49,99.99,194.47,369.00,1269.00
Storage & Memory,1426.00,438.26,922.65,3.79,86.98,179.99,419.74,10212.99
"Peripherals, Smart Home & Gadgets",899.00,182.62,326.86,2.99,44.99,79.99,179.99,3199.90


**What we see:**
- The groups have clearly different price levels: Mac & Computers (median about €2,700), NAS & Servers (about €980), iPhones (about €770), iPads (about €730), down to Cases, Cables & Accessories (about €40).  
- iPhones & Smartphones are the cleanest: nothing under €409.
- iPads & Tablets only contains iPads: the cheapest is an open-box iPad mini at €239. Moving the Wacom tablets to Peripherals removed the €79 products.
- Peripherals, Smart Home & Gadgets now has the Wacom tablets, so its maximum is the Wacom MobileStudio Pro (€3,200). That one is a full pen computer, unlike the other Wacom tablets, but there are only a few.
- NAS & Servers has a few cheap add-ons (RAM, drive trays, rail kits, licences) from the NAS accessories types. They are few, so the median isn't affected.
- Storage & Memory has the widest spread (€4 memory cards to €10,000 RAID drives). Same kind of product, very different sizes.
- *Unsorted (mostly open box)* has 931 products, median €61. The last cell shows that 698 of the open-box products end up there; the other open-box products are devices and monitors.

In [45]:
# how many sub-categories each group has
products_df.groupby("grouped_category")["sub_category"].nunique().sort_values(ascending=False)

grouped_category
Cases, Cables & Accessories          21
Mac & Computers                      11
iPhones & Smartphones                11
iPads & Tablets                      11
Peripherals, Smart Home & Gadgets     9
Storage & Memory                      8
Audio, Wearables & Media Players      6
Repairs, Parts & Services             4
NAS & Servers                         2
Monitors & Displays                   1
Others                                1
Unsorted (mostly open box)            1
Name: sub_category, dtype: int64

In [46]:
# where the open-box products of type 1298 ended up
products_df.loc[products_df["type"] == "1298", "grouped_category"].value_counts()

grouped_category
Unsorted (mostly open box)          698
Monitors & Displays                  38
Mac & Computers                      31
iPads & Tablets                      10
iPhones & Smartphones                 5
Audio, Wearables & Media Players      1
Name: count, dtype: int64

In [47]:
products_df.head(20)

,sku,name,desc,price,in_stock,type,sub_category,condition,grouped_category
0,RAI0007,Silver Rain Design mStand Support,Aluminum support compatible with all MacBook,59.99,1,8696,Laptop & Desktop Stands,New,"Cases, Cables & Accessories"
1,APP0023,Apple Mac Keyboard Keypad Spanish,USB ultrathin keyboard Apple Mac Spanish.,59.00,0,13855401,Keyboards,New,"Peripherals, Smart Home & Gadgets"
2,APP0025,Mighty Mouse Apple Mouse for Mac,mouse Apple USB cable.,59.00,0,1387,Mice & Trackpads,New,"Peripherals, Smart Home & Gadgets"
3,APP0072,Apple Dock to USB Cable iPhone and iPod white,IPhone dock and USB Cable Apple iPod.,25.00,0,1230,Lightning Cables,New,"Cases, Cables & Accessories"
4,KIN0007,Mac Memory Kingston 2GB 667MHz DDR2 SO-DIMM,2GB RAM Mac mini and iMac (2006/07) MacBook Pr...,34.99,1,1364,RAM Memory,New,Storage & Memory
5,APP0073,Apple Composite AV Cable iPhone and iPod white,IPhone and iPod AV Cable Dock to Composite Video.,45.00,0,1230,Lightning Cables,New,"Cases, Cables & Accessories"
6,KIN0008,Mac Memory Kingston 1GB 667MHz DDR2 SO-DIMM,1GB RAM Mac mini and iMac (2006/07) MacBook Pr...,18.99,0,1364,RAM Memory,New,Storage & Memory
7,KIN0009,Mac Memory Kingston 2GB 800MHz DDR2 SO-DIMM,2GB RAM iMac with Intel Core 2 Duo (Penryn).,36.99,0,1364,RAM Memory,New,Storage & Memory
8,KIN0001-2,Mac memory Kingston 4GB (2x2GB) 667MHz DDR2 SO...,RAM 4GB (2x2GB) Mac mini and iMac (2006/07) Ma...,74.00,0,1364,RAM Memory,New,Storage & Memory
9,APP0100,Apple Adapter Mini Display Port to VGA,Adapter Mini Display Port to VGA MacBook and M...,35.00,0,1325,"Video Cables, Adapters & Locks",New,"Cases, Cables & Accessories"


## 9. Export

In [48]:
# save the categorised products as a CSV file
products_df.to_csv("products_categorised.csv", index=False)

In [ ]:
# download it from Colab (then upload to data/clean-data)
from google.colab import files
files.download("products_categorised.csv")